# Formaldehyde — TDDFT, TDDFT+SOC, TDDFT+SOC+QED

Pipeline for H₂CO:

1. **Bare TDDFT (TDA)** — closed-shell singlets and triplets
2. **TDDFT + SI-SOC** — one-electron state-interaction mixing
3. **TDDFT + SOC + QED** — truncated Pauli–Fierz on SI-SOC ⊗ {0,1} photons (borrowed dipoles + DSE);
   Tavis–Cummings remains available as ``solve_soc_qed_levels``

Compare to **singlet-only QED-TDA** (no SOC) to see how triplets enter the polariton manifold.

Level A: ordinary RKS ground state; SOC and cavity are post-SCF / response-level.

Defaults: `sto-3g` + `pbe` for quick runs. Bump basis / XC for production.


In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from pyscf import gto, dft

from casidapy import (
    extract_gto_kernel,
    run_casida,
    solve_soc_si,
    solve_soc_qed_levels,
    solve_soc_qed_pf,
    solve_qed_tda,
    QEDOptions,
)

HA_TO_EV = 27.211386245988
XC = "pbe"
BASIS = "sto-3g"
N_S = 4
N_T = 4
LAM = 0.05
POL = (0.0, 0.0, 1.0)  # along C=O (z)

FORMALDEHYDE = """
C  0.000000  0.000000  0.000000
O  0.000000  0.000000  1.208000
H  0.000000  0.943000 -0.587000
H  0.000000 -0.943000 -0.587000
"""
print(f"defaults: {XC}/{BASIS}, λ={LAM}, pol=z")


## 1. SCF + bare TDDFT (singlet & triplet TDA)


In [ ]:
mol = gto.M(atom=FORMALDEHYDE, basis=BASIS, verbose=0)
mf = dft.RKS(mol)
mf.xc = XC
mf.grids.level = 1
e_scf = mf.kernel()
print(f"SCF E = {e_scf:.8f} Ha")

ks, opts_s = extract_gto_kernel(
    mf, n_states=N_S, tda=True, use_df=False, spin_state="singlet",
)
kt, opts_t = extract_gto_kernel(
    mf, n_states=N_T, tda=True, use_df=False, spin_state="triplet",
)
opts_s.solver_method = opts_t.solver_method = "eigsh"
res_s = run_casida(ks, opts_s)
res_t = run_casida(kt, opts_t)

print("\nSinglet TDA:")
for i, w in enumerate(res_s.omega):
    print(f"  S{i+1}: {w*HA_TO_EV:8.3f} eV   f={res_s.f[i]:.4e}")
print("Triplet TDA:")
for i, w in enumerate(res_t.omega):
    print(f"  T{i+1}: {w*HA_TO_EV:8.3f} eV   f={res_t.f[i]:.4e}")


## 2. TDDFT + SI-SOC


In [ ]:
soc = solve_soc_si(res_s, res_t, ks, include_ground=False)

print("SI-SOC mixed roots:")
print(f"{'i':>4} {'ω (eV)':>10} {'S wt':>8} {'T wt':>8} {'f':>10}")
print("-" * 44)
for i, w in enumerate(soc.omega):
    print(
        f"{i:4d} {w*HA_TO_EV:10.3f} {soc.singlet_weight[i]:8.3f} "
        f"{soc.triplet_weight[i]:8.3f} {soc.f[i]:10.4e}"
    )


## 3. QED — singlets only vs SOC-mixed few-level

- **QED-TDA (no SOC):** full CasidaPy Pauli–Fierz TDA on the singlet manifold
- **SOC+QED:** Pauli–Fierz on SI-SOC ⊗ {0,1} photons (bilinear + DSE) using borrowed dipoles

Cavity frequency defaults to the lowest singlet root (Ha).


In [ ]:
omega_c = float(res_s.omega[int(np.argmax(res_s.f))])
print(f"ω_c = {omega_c*HA_TO_EV:.3f} eV (brightest singlet)")

qed_opts = QEDOptions(
    lam_scalar=LAM, polarization=POL, omega_c=omega_c, nstates=8,
)
qed_singlet = solve_qed_tda(ks, options=qed_opts)

lam_vec = np.asarray(POL, float) * LAM
qed_soc = solve_soc_qed_pf(
    soc,
    lam_vec=lam_vec,
    omega_c=omega_c,
    nstates=None,
    include_dse=True,
)

print("\nQED-TDA (singlets only):")
for i, w in enumerate(qed_singlet.omega[:8]):
    print(f"  {i}: {w*HA_TO_EV:8.3f} eV   |m|²={qed_singlet.photon_frac[i]:.3f}")
print("SOC+PF-QED (electronic ⊗ {0,1}):")
for i, w in enumerate(qed_soc["omega"][:12]):
    print(f"  {i}: {w*HA_TO_EV:8.3f} eV   |m|²={qed_soc['photon_frac'][i]:.3f}")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.8), sharey=False)

# Bare electronic
ax = axes[0]
ax.vlines(res_s.omega * HA_TO_EV, 0, np.maximum(res_s.f, 1e-6), colors="C0", lw=1.6, label="S")
ax.vlines(res_t.omega * HA_TO_EV, 0, 0.04, colors="C1", lw=1.2, label="T")
ax.set_title("Bare TDA")
ax.set_xlabel("ω (eV)")
ax.set_ylabel("f / arb")
ax.legend(fontsize=8)

# SOC
ax = axes[1]
fplot = np.maximum(soc.f, 1e-8)
sc = ax.scatter(soc.omega * HA_TO_EV, fplot, c=soc.triplet_weight, cmap="coolwarm",
                vmin=0, vmax=1, s=45, zorder=3)
ax.vlines(soc.omega * HA_TO_EV, 0, fplot, colors="0.75", lw=1.0)
ax.set_title("TDA + SI-SOC")
ax.set_xlabel("ω (eV)")
fig.colorbar(sc, ax=ax, label="triplet wt", fraction=0.046)

# QED comparison
ax = axes[2]
ax.vlines(qed_singlet.omega[:8] * HA_TO_EV, 0, 1.0, colors="C0", lw=1.4, label="QED (S only)")
ax.vlines(qed_soc["omega"] * HA_TO_EV, 0, 0.7, colors="C3", lw=1.4, alpha=0.85, label="SOC+QED")
ax.axvline(omega_c * HA_TO_EV, color="k", ls=":", lw=1, label="ω_c")
ax.set_title(f"QED (λ={LAM})")
ax.set_xlabel("ω (eV)")
ax.set_ylabel("arb")
ax.legend(fontsize=7)

fig.suptitle(f"Formaldehyde — {XC}/{BASIS}", y=1.03)
fig.tight_layout()
plt.show()


## PES scan

For a C=O stretch PES comparing all three electronic/polariton ladders, run:

```bash
python scripts/plot_formaldehyde_soc_qed_pes.py
python scripts/plot_formaldehyde_soc_qed_pes.py --npoints 9 --out formaldehyde_soc_qed_pes.png
```

That script writes a PNG (and optional `.npz`) with total energies \(E_\mathrm{SCF}+\omega\) along the stretch.
